In [20]:
from datasets import load_dataset

ds = load_dataset("mychen76/invoices-and-receipts_ocr_v1")

/opt/anaconda3/envs/agent/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
test = ds['train']

In [87]:
import json
parsed_row = json.loads(test[3]['parsed_data'])
parsed_row['json']

'{\'header\': {\'invoice_no\': \'27301261\', \'invoice_date\': \'10/09/2012\', \'seller\': \'Williams LLC 72074 Taylor Plains Suite 342 West Alexandria, AR 97978\', \'client\': \'Hernandez-Anderson 084 Carter Lane Apt. 846 South Ronaldbury, AZ 91030\', \'seller_tax_id\': \'922-88-2832\', \'client_tax_id\': \'959-74-5868\', \'iban\': \'GB70FTNR64199348221780\'}, \'items\': [{\'item_desc\': \'Lilly Pulitzer dress Size 2\', \'item_qty\': \'5,00\', \'item_net_price\': \'45,00\', \'item_net_worth\': \'225,00\', \'item_vat\': \'10%\', \'item_gross_worth\': \'247,50\'}, {\'item_desc\': \'New ERIN Erin Fertherston Straight Dress White Sequence Lining Sleeveless SZ 10\', \'item_qty\': \'1,00\', \'item_net_price\': \'59,99\', \'item_net_worth\': \'59,99\', \'item_vat\': \'10%\', \'item_gross_worth\': \'65,99\'}, {\'item_desc\': \'Sequence dress Size Small\', \'item_qty\': \'3,00\', \'item_net_price\': \'35,00\', \'item_net_worth\': \'105,00\', \'item_vat\': \'10%\', \'item_gross_worth\': \'115,5

In [ ]:
from PIL import Image, ImageOps

img = test[1]['image']
max_size = (800, 800)

resized_img = ImageOps.contain(img, max_size, Image.Resampling.LANCZOS)
# resized_img.save('output.jpg')


In [ ]:
import instructor
from pydantic import BaseModel, field_validator
from datetime import date, datetime
from schemas import Invoice
from openai import OpenAI

from dotenv import load_dotenv
import os

load_dotenv()

gemini = os.getenv('GEMINI_KEY')
openai = os.getenv('OPEN_AI_API')

# instructor wraps the client so `response_model=Invoice` validates the model's
# output straight into our Pydantic schema (retrying on validation errors).
client = instructor.from_openai(OpenAI(api_key=openai))

# One place to set the extraction model. gpt-4o-mini is cheap and reliable for
# structured extraction; swap to a nano/smaller tier if you want lower cost.
EXTRACT_MODEL = "gpt-5.4-nano"

# 3. Feed the OCR'd TEXT as the prompt, ask for the schema back.
ocr_text = """Invoice no: 61356291
Date of issue: 09/06/2012

Seller: Client:
Chapman, Kim and Green
64731 James Branch
Smithmouth, NC 26872

Rodriguez-Stevens
2280 Angela Plain
Hortonshire, MS 93248

Tax Id: 949-84-9105
IBAN: GB50ACIE59715038217063

ITEMS
No. Description Qty UM Net price Net worth VAT [%] Gross worth
1. Wine Glasses Goblets Pair Clear Glass 5.00 each 12.00 60.00 10% 66.00
2. With Hooks Stemware Storage Multiple Uses Iron Wine Rack Hanging Glass 4.00 each 28.08 112.32 10% 123.55
3. Replacement Corkscrew Parts Spiral Worm Wine Opener Bottle Houdini 1.00 each 7.50 7.50 10% 8.25
4. HOME ESSENTIALS GRADIENT STEMLESS WINE GLASSES SET OF 4 20 FL OZ (591 ml) NEW

SUMMARY
VAT [%] Net worth VAT Gross worth
10% 192.81 19.28 212.09
Total $192,81 $19,28 $212,09"""

invoice = client.chat.completions.create(
    model=EXTRACT_MODEL,
    response_model=Invoice,                # <-- this is the whole instructor mechanism
    messages=[{
        "role": "user",
        "content": f"Extract the invoice fields from this text :\n\n{ocr_text}",
    }],
)

In [45]:
for data in invoice:
    print(data)

('header', Header(invoice_no='61356291', invoice_date='09/06/2012', seller='Chapman, Kim and Green\n64731 James Branch\nSmithmouth, NC 26872', client='Rodriguez-Stevens\n2280 Angela Plain\nHortonshire, MS 93248', seller_tax_id=None, client_tax_id=None, iban='GB50ACIE59715038217063'))
('items', [LineItem(item_desc='Wine Glasses Goblets Pair Clear Glass', item_qty=5, item_net_price=Decimal('12.00'), item_net_worth=Decimal('60.00'), item_vat='10%', item_gross_worth=Decimal('66.00')), LineItem(item_desc='With Hooks Stemware Storage Multiple Uses Iron Wine Rack Hanging Glass', item_qty=4, item_net_price=Decimal('28.08'), item_net_worth=Decimal('112.32'), item_vat='10%', item_gross_worth=Decimal('123.55')), LineItem(item_desc='Replacement Corkscrew Parts Spiral Worm Wine Opener Bottle Houdini', item_qty=1, item_net_price=Decimal('7.50'), item_net_worth=Decimal('7.50'), item_vat='10%', item_gross_worth=Decimal('8.25')), LineItem(item_desc='HOME ESSENTIALS GRADIENT STEMLESS WINE GLASSES SET OF

In [10]:
# Canonical DDL now lives in schema.sql — single source of truth for BOTH the
# ingest path (this notebook) and the agent (db.py / tools.py). Edit schema.sql,
# never inline strings, so the two halves can never drift apart again.
with open("schema.sql") as f:
    schema_sql = f.read()
print(schema_sql[:300], "...")

-- Canonical schema for the OCR -> Supabase -> RAG pipeline.
-- Single source of truth: prepare_db.ipynb (ingest) and db.py/tools.py (agent) both target this.
-- Embeddings are OpenAI text-embedding-3-small => vector(1536).
-- Most columns are nullable on purpose: OCR/extraction can miss fields, and ...


In [ ]:
# Reset = wipe Storage images + DROP/recreate the tables from schema.sql (the single
# source of truth). pipeline.py owns this logic; the notebook just drives it.
# WARNING: destroys every ingested row + uploaded image — only run before a full re-seed.
import pipeline

pipeline.reset()

**Retired cell.** The notebook used to define its own `insert_data()` here (and its own storage/OCR helpers further down). That duplicate ingest path is what let the notebook drift from the backend and pair OCR text with the wrong ground truth. All ingest logic now lives in `pipeline.py` — `upload() → ocr() → extract() → insert()` — and both this notebook and the FastAPI backend (`main.py`) call the same `pipeline.ingest_document()`, so they cannot drift.

In [ ]:
# Bulk seed the DB through the SAME path the API uses (pipeline.ingest_document),
# so the notebook and the backend can't drift. Each call does the whole flow for one
# dataset image: upload -> OCR -> extract -> insert. source_id = "ds_{i}" lets the live
# demo later avoid re-ingesting these. Run the schema setup cell above first.
import pipeline
from datasets import load_dataset

ds = load_dataset("mychen76/invoices-and-receipts_ocr_v1")["train"]

N = 150
ok = fail = 0
for i in range(N):
    try:
        pipeline.ingest_document(ds[i]["image"], source_id=f"ds_{i}")
        ok += 1
    except Exception as e:
        fail += 1
        print(f"[{i}] failed: {str(e)[:120]}")
    if ok and ok % 10 == 0:
        print(f"seeded {ok}/{N}")
print(f"done: seeded {ok}, failed {fail}")

In [ ]:
# Build the evaluation set. The logic lives in build_eval_set.py (single source of
# truth — same pattern as pipeline.py) so the notebook and CI can't drift. Rows are
# {question, ground_truth, type, target_invoices}; ground truth comes from CLEAN
# parsed_data (never OCR text), and target_invoices enables judge-free retrieval
# metrics (hit@5 / MRR). The set is frozen at the v0-baseline tag — every version
# re-runs the same questions so deltas are apples-to-apples.
import build_eval_set

build_eval_set.main()

**Retired: the `processed_data.json` path.** Earlier versions of this notebook OCR'd the local `./images/{i}.jpg` files and cached the text in `processed_data.json` next to `parsed_data[i]`. Those local files were **not** the same documents as `parsed_data[i]`, so OCR text and ground truth described different documents — that misalignment is what flooded the DB with empty/wrong rows.

The fix is structural, not a patch: the seeding cell above OCRs `ds[i]["image"]` **directly** through `pipeline.ingest_document(..., source_id=f"ds_{i}")`, so image, OCR text, and stored row are aligned by construction, and `source_id` maps each DB row back to its dataset ground truth. `evaluate_extraction.py` reads `raw_ocr` straight from the DB and joins ground truth via `source_id` — `processed_data.json` is no longer used by anything and can be deleted.

In [20]:
from db import embed_query,get_conn
query= "Food"
query_vec = embed_query(query)
with get_conn() as conn, conn.cursor() as cur:
    # <=> is pgvector's cosine-distance operator; smaller = more similar.
    # ::vector — psycopg sends a Python list as float8[], and no <=> operator
    # exists for (vector, float8[]); the explicit cast resolves it.
    cur.execute(
        "select content from chunks order by embedding <=> %s::vector limit %s",
        (query_vec, 3),
    )
    hits = [r[0] for r in cur.fetchall()]

In [22]:
from sentence_transformers import CrossEncoder

# Load the BGE English reranker model
model = CrossEncoder('BAAI/bge-reranker-large', max_length=512)



# Create query-document pairs
pairs = [[query, doc] for doc in hits]

# Compute scores
scores = model.predict(pairs)
for doc, score in zip(hits, scores):
    print(f"Score: {score:.4f} - Document: {doc}")


ModuleNotFoundError: No module named 'sentence_transformers'

In [ ]:
#here we test PaddleOCR
#load picture
#run ocr
#extract Data
#fill pydantic structure

In [ ]:
#initialize Supabase client 
#form connection
#create table


In [ ]:
#prepare data for ingestion 
#set metadata for table and embed content
#normalization makes it easier to retrieve
# later we just numpys @ dot product , and sort it and get top results
#oh we might need to go to ragas to evaluate retrieval NDCG might be good evaluation method
#also we just might not always need hybrid could only be bm25+reranker  or embedding + reranker or could bbe hybrid we have to 
#decide which one is better in this case
#as we dont have readily available evalset we can use llm to generate eval questsions
#then use it to evaluate retrieval system

#ingest data to db

In [ ]:
#retrieve data with naive search and demonstrate different embedding models quality
#then on best model we make it hybrid search +reranker
#rerank retrieved information

In [ ]:
#prepare prompts
#define agent
#define retrieval tools
#test agent

In [ ]:
#prepare ragas questsions
#prepare github actions
#evaluate model with Ragas
#log result


In [ ]:
#but first we define simple embedeed retrieval no hybrid search reranker
#then same model but retrieval improved with hybrid and reranker
#then change model compare results on same prompt

#prompt engineering to improve model result

#cache model answer for similiar questsions to lower api calls for same questsions 
